# Ghost Writer — Training Notebook

**Phase 1**: Word-level classifier (CNN + GlobalAvgPool) — proves accelerometer signal can distinguish words  
**Phase 2**: Character-level CTC recognizer (CNN + BiLSTM + CTC) — decodes arbitrary text

Upload `training/` folder and `training_data/` directory to Colab, then run all cells.  
Samples are loaded from all session files in `training_data/sessions/` plus the legacy `training_data/samples.jsonl`.

In [ ]:
import sys, os

# On Colab, install deps if needed
if "google.colab" in sys.modules:
    os.system("pip install -q torch numpy matplotlib")

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# --- Configuration ---
DATA_PATH = "training_data"  # directory — loads all session files + legacy samples.jsonl
PHASE = 1  # Set to 1 for word classifier, 2 for CTC character decoder

# Phase 1 hyperparameters
P1_EPOCHS = 80
P1_LR = 1e-3
P1_BATCH = 16

# Phase 2 hyperparameters
P2_EPOCHS = 120
P2_LR = 5e-4
P2_BATCH = 16

# Train/val split
VAL_FRACTION = 0.2
AUGMENT_TRAIN = True

## Load Dataset & Inspect

In [ ]:
from training.dataset import load_all_samples, get_all_stats
from training.data_pipeline import (
    WordDataset, CTCDataset,
    collate_word, collate_ctc,
    compute_features,
)
from training.model import (
    WordClassifier, CTCRecognizer,
    encode_text, decode_ctc, NUM_CLASSES,
)

# Show dataset stats (across all session files + legacy)
stats = get_all_stats(DATA_PATH)
print(f"Total samples: {stats['total_samples']}")
print(f"Total duration: {stats['total_duration_s']:.1f}s")
print(f"\nWords ({len(stats['words'])} unique):")
for word, count in sorted(stats["words"].items(), key=lambda x: -x[1]):
    print(f"  {word:15s} {count:4d} samples")

In [ ]:
# Visualize a sample's features
raw = load_all_samples(DATA_PATH)
if raw:
    sample = raw[0]
    feat = compute_features(sample["samples"])
    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
    t = np.arange(len(feat))

    axes[0].plot(t, feat[:, 0], label="x")
    axes[0].plot(t, feat[:, 1], label="y")
    axes[0].plot(t, feat[:, 2], label="z")
    axes[0].set_ylabel("Raw accel (g)")
    axes[0].legend()
    axes[0].set_title(f'Word: "{sample["word"]}" — {len(feat)} timesteps')

    axes[1].plot(t, feat[:, 3], label="dx")
    axes[1].plot(t, feat[:, 4], label="dy")
    axes[1].plot(t, feat[:, 5], label="dz")
    axes[1].set_ylabel("Deltas")
    axes[1].legend()

    axes[2].plot(t, feat[:, 6], label="L2 norm", color="black")
    axes[2].set_ylabel("L2 norm")
    axes[2].legend()

    axes[3].plot(t, feat[:, 7], label="ddx")
    axes[3].plot(t, feat[:, 8], label="ddy")
    axes[3].plot(t, feat[:, 9], label="ddz")
    axes[3].set_ylabel("2nd-order deltas")
    axes[3].set_xlabel("Timestep")
    axes[3].legend()

    plt.tight_layout()
    plt.show()
else:
    print("No samples yet — collect data with the training UI first!")

## Phase 1: Word-Level Classifier

Proves the accelerometer signal contains enough information to distinguish words. This is your feasibility gate — if this doesn't work, character-level decoding won't either.

In [ ]:
if PHASE == 1:
    # Build dataset and split
    full_ds = WordDataset(DATA_PATH, augment_data=False)
    print(f"Vocabulary: {full_ds.word_to_idx}")
    print(f"Total samples: {len(full_ds)}, Num words: {full_ds.num_words}")

    val_size = max(1, int(len(full_ds) * VAL_FRACTION))
    train_size = len(full_ds) - val_size
    train_ds_raw, val_ds = random_split(full_ds, [train_size, val_size],
                                        generator=torch.Generator().manual_seed(SEED))

    # Wrap train split with augmentation
    class AugmentedSubset(torch.utils.data.Dataset):
        def __init__(self, subset, filepath, word_to_idx):
            self.subset = subset
            self.aug_ds = WordDataset(filepath, word_to_idx=word_to_idx,
                                     augment_data=AUGMENT_TRAIN)
        def __len__(self):
            return len(self.subset)
        def __getitem__(self, idx):
            orig_idx = self.subset.indices[idx]
            return self.aug_ds[orig_idx]

    train_ds = AugmentedSubset(train_ds_raw, DATA_PATH, full_ds.word_to_idx)

    train_loader = DataLoader(train_ds, batch_size=P1_BATCH, shuffle=True,
                              collate_fn=collate_word, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=P1_BATCH, shuffle=False,
                            collate_fn=collate_word, drop_last=False)

    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
    print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
if PHASE == 1:
    model = WordClassifier(num_features=10, num_words=full_ds.num_words).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=P1_LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=P1_EPOCHS)
    criterion = nn.CrossEntropyLoss()

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")
    print(model)

In [ ]:
if PHASE == 1:
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(1, P1_EPOCHS + 1):
        # --- Train ---
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for features, labels, lengths in train_loader:
            features, labels, lengths = features.to(device), labels.to(device), lengths.to(device)
            optimizer.zero_grad()
            logits = model(features, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()
        train_losses.append(total_loss / total)
        train_accs.append(correct / total)

        # --- Validate ---
        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels, lengths in val_loader:
                features, labels, lengths = features.to(device), labels.to(device), lengths.to(device)
                logits = model(features, lengths)
                loss = criterion(logits, labels)
                total_loss += loss.item() * labels.size(0)
                correct += (logits.argmax(dim=1) == labels).sum().item()
                total += labels.size(0)
        val_losses.append(total_loss / total)
        val_accs.append(correct / total)

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{P1_EPOCHS}  "
                  f"train_loss={train_losses[-1]:.4f}  train_acc={train_accs[-1]:.3f}  "
                  f"val_loss={val_losses[-1]:.4f}  val_acc={val_accs[-1]:.3f}  "
                  f"lr={scheduler.get_last_lr()[0]:.6f}")

    print(f"\nBest val accuracy: {max(val_accs):.3f} at epoch {np.argmax(val_accs)+1}")

In [ ]:
if PHASE == 1:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(train_losses, label="Train")
    ax1.plot(val_losses, label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Cross-Entropy Loss")
    ax1.legend()

    ax2.plot(train_accs, label="Train")
    ax2.plot(val_accs, label="Val")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Word Classification Accuracy")
    ax2.axhline(y=1/full_ds.num_words, color="gray", linestyle="--", label="Random chance")
    ax2.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Confusion matrix for Phase 1
if PHASE == 1:
    from collections import Counter

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for features, labels, lengths in val_loader:
            features, lengths = features.to(device), lengths.to(device)
            logits = model(features, lengths)
            preds = logits.argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    # Print per-word results
    print("Validation predictions:")
    for pred, true in zip(all_preds, all_labels):
        pred_word = full_ds.idx_to_word[pred]
        true_word = full_ds.idx_to_word[true]
        mark = "✓" if pred == true else "✗"
        print(f"  {mark}  true={true_word:12s}  pred={pred_word}")

    acc = sum(p == t for p, t in zip(all_preds, all_labels)) / len(all_labels)
    print(f"\nOverall val accuracy: {acc:.1%}")

## Phase 2: Character-Level CTC Recognizer

Only run this after Phase 1 shows >70% word accuracy, confirming the signal is rich enough. Set `PHASE = 2` in the config cell above and re-run from there.

In [ ]:
if PHASE == 2:
    full_ds_ctc = CTCDataset(DATA_PATH, augment_data=False)
    print(f"Total samples: {len(full_ds_ctc)}")

    val_size = max(1, int(len(full_ds_ctc) * VAL_FRACTION))
    train_size = len(full_ds_ctc) - val_size
    train_ds_ctc_raw, val_ds_ctc = random_split(
        full_ds_ctc, [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    # Augmented train wrapper
    class AugCTCSubset(torch.utils.data.Dataset):
        def __init__(self, subset, filepath):
            self.subset = subset
            self.aug_ds = CTCDataset(filepath, augment_data=AUGMENT_TRAIN)
        def __len__(self):
            return len(self.subset)
        def __getitem__(self, idx):
            return self.aug_ds[self.subset.indices[idx]]

    train_ds_ctc = AugCTCSubset(train_ds_ctc_raw, DATA_PATH)

    train_loader = DataLoader(train_ds_ctc, batch_size=P2_BATCH, shuffle=True,
                              collate_fn=collate_ctc, drop_last=False)
    val_loader = DataLoader(val_ds_ctc, batch_size=P2_BATCH, shuffle=False,
                            collate_fn=collate_ctc, drop_last=False)

    print(f"Train: {len(train_ds_ctc)}, Val: {len(val_ds_ctc)}")

In [ ]:
if PHASE == 2:
    model = CTCRecognizer(num_features=10, hidden_size=128, num_layers=2,
                          dropout=0.3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=P2_LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=P2_EPOCHS)
    ctc_loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")
    print(model)

In [ ]:
if PHASE == 2:
    train_losses, val_losses = [], []

    for epoch in range(1, P2_EPOCHS + 1):
        # --- Train ---
        model.train()
        total_loss, total_samples = 0.0, 0
        for features, targets, input_lengths, target_lengths in train_loader:
            features = features.to(device)
            targets = targets.to(device)
            input_lengths = input_lengths.to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            log_probs, output_lengths = model(features, input_lengths)
            loss = ctc_loss_fn(log_probs, targets, output_lengths, target_lengths)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item() * features.size(0)
            total_samples += features.size(0)
        scheduler.step()
        train_losses.append(total_loss / total_samples)

        # --- Validate ---
        model.eval()
        total_loss, total_samples = 0.0, 0
        with torch.no_grad():
            for features, targets, input_lengths, target_lengths in val_loader:
                features = features.to(device)
                targets = targets.to(device)
                input_lengths = input_lengths.to(device)
                target_lengths = target_lengths.to(device)
                log_probs, output_lengths = model(features, input_lengths)
                loss = ctc_loss_fn(log_probs, targets, output_lengths, target_lengths)
                total_loss += loss.item() * features.size(0)
                total_samples += features.size(0)
        val_losses.append(total_loss / total_samples)

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{P2_EPOCHS}  "
                  f"train_loss={train_losses[-1]:.4f}  "
                  f"val_loss={val_losses[-1]:.4f}  "
                  f"lr={scheduler.get_last_lr()[0]:.6f}")

    print(f"\nBest val loss: {min(val_losses):.4f} at epoch {np.argmin(val_losses)+1}")

In [ ]:
if PHASE == 2:
    # Loss curves
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_losses, label="Train")
    ax.plot(val_losses, label="Val")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("CTC Loss")
    ax.set_title("CTC Training Progress")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# CTC greedy decode on validation set
if PHASE == 2:
    model.eval()
    raw_samples = load_samples(DATA_PATH)

    print("Validation predictions (greedy CTC decode):\n")
    with torch.no_grad():
        for features, targets, input_lengths, target_lengths in val_loader:
            features = features.to(device)
            input_lengths = input_lengths.to(device)
            log_probs, output_lengths = model(features, input_lengths)

            # Greedy decode each sample in the batch
            pred_indices = log_probs.argmax(dim=2).transpose(0, 1)  # (batch, time)
            offset = 0
            for i in range(features.size(0)):
                # Decode prediction
                pred_text = decode_ctc(pred_indices[i, :output_lengths[i]].tolist())

                # Decode target
                tgt_len = target_lengths[i].item()
                tgt_indices = targets[offset:offset + tgt_len].tolist()
                from training.model import IDX_TO_CHAR
                true_text = "".join(IDX_TO_CHAR.get(idx, "?") for idx in tgt_indices)
                offset += tgt_len

                mark = "✓" if pred_text == true_text else "✗"
                print(f"  {mark}  true={true_text:15s}  pred={pred_text}")

    print("\nDone!")

## Save Model

In [ ]:
# Save trained model
save_path = f"ghost_writer_phase{PHASE}.pt"
save_dict = {
    "model_state_dict": model.state_dict(),
    "phase": PHASE,
}
if PHASE == 1:
    save_dict["word_to_idx"] = full_ds.word_to_idx
    save_dict["idx_to_word"] = full_ds.idx_to_word

torch.save(save_dict, save_path)
print(f"Model saved to {save_path}")